In [ ]:
import hashlib
import secrets

# secp256k1 parameters — the numbers that secure Bitcoin

# Field prime: coordinates live in F_P
SECP_P = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F

# Group order: scalars (private keys) live in Z_N  
SECP_N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

# Generator point G: the "starting point" for all key generation
SECP_GX = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
SECP_GY = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8

# Curve coefficients
A_COEFF = 0
B_COEFF = 7

print("=== secp256k1 Parameters ===")
print(f"Curve: y² = x³ + {A_COEFF}x + {B_COEFF}")
print(f"P = 2²⁵⁶ - 2³² - 977")
print(f"  = {SECP_P}")
print(f"  ({SECP_P.bit_length()} bits)")
print(f"\nN = {SECP_N}")
print(f"  ({SECP_N.bit_length()} bits)")
print(f"\nNon-singular check: 4(0)³ + 27(7)² = {4*0**3 + 27*7**2} ≠ 0  ✓")
print(f"\nP mod 4 = {SECP_P % 4}  (enables efficient square roots)")

from typing import Optional

class Point:
    """A point on secp256k1 (or infinity)."""
    def __init__(self, x: Optional[int] = None, y: Optional[int] = None):
        self.x = x
        self.y = y
    
    def is_infinity(self) -> bool:
        return self.x is None or self.y is None
    
    def copy(self) -> 'Point':
        return Point(self.x, self.y)
    
    def __eq__(self, other):
        if self.is_infinity() and other.is_infinity():
            return True
        return self.x == other.x and self.y == other.y
    
    def __repr__(self):
        if self.is_infinity():
            return "O (point at infinity)"
        return f"({hex(self.x)[:16]}..., {hex(self.y)[:16]}...)"

G = Point(SECP_GX, SECP_GY)
INFINITY = Point()  # The identity element

print(f"Generator G:     {G}")
print(f"Identity O:      {INFINITY}")

def mod_inverse(a: int, p: int) -> int:
    """a⁻¹ mod p via Fermat's little theorem: a^(p-2) mod p."""
    return pow(a, p - 2, p)

def mod_sqrt(a: int, p: int) -> int:
    """√a mod p for p ≡ 3 (mod 4): a^((p+1)/4) mod p."""
    return pow(a, (p + 1) // 4, p)

def point_add(p1: Point, p2: Point) -> Point:
    """
    Add two distinct points on secp256k1.
    
    Slope:  λ = (y₂ - y₁) / (x₂ - x₁) mod P
    Result: x₃ = λ² - x₁ - x₂
            y₃ = λ(x₁ - x₃) - y₁
    """
    if p1.is_infinity():
        return p2.copy()
    if p2.is_infinity():
        return p1.copy()
    
    if p1.x == p2.x:
        if (p1.y + p2.y) % SECP_P == 0:
            return Point()  # P + (-P) = O
        return point_double(p1)
    
    lam = ((p2.y - p1.y) * mod_inverse(p2.x - p1.x, SECP_P)) % SECP_P
    x3 = (lam * lam - p1.x - p2.x) % SECP_P
    y3 = (lam * (p1.x - x3) - p1.y) % SECP_P
    return Point(x3, y3)

def point_double(p: Point) -> Point:
    """
    Double a point on secp256k1.
    
    Slope:  λ = 3x² / 2y mod P  (tangent to curve at P)
    Result: x₃ = λ² - 2x
            y₃ = λ(x - x₃) - y
    """
    if p.is_infinity() or p.y == 0:
        return Point()
    
    lam = (3 * p.x * p.x * mod_inverse(2 * p.y, SECP_P)) % SECP_P
    x3 = (lam * lam - 2 * p.x) % SECP_P
    y3 = (lam * (p.x - x3) - p.y) % SECP_P
    return Point(x3, y3)

def point_negate(p: Point) -> Point:
    """Negate: -P = (x, -y mod P)."""
    if p.is_infinity():
        return Point()
    return Point(p.x, (SECP_P - p.y) % SECP_P)

# Verify G is on the curve
lhs = (G.y * G.y) % SECP_P
rhs = (G.x ** 3 + 7) % SECP_P
print(f"G on curve? y² mod P == x³+7 mod P: {lhs == rhs}  ✓")

# Verify group properties
G2 = point_double(G)
print(f"\n2G = {G2}")
print(f"G + O = G? {point_add(G, INFINITY) == G}  ✓  (identity)")
neg_G = point_negate(G)
print(f"G + (-G) = O? {point_add(G, neg_G).is_infinity()}  ✓  (inverse)")

def scalar_mult(k: int, p: Point) -> Point:
    """Compute k × P using double-and-add. O(log k) operations."""
    if k == 0 or p.is_infinity():
        return Point()
    k = k % SECP_N
    if k == 0:
        return Point()
    
    result = Point()  # Start at O (identity)
    addend = p.copy()
    
    while k > 0:
        if k & 1:
            result = point_add(result, addend)
        addend = point_double(addend)
        k >>= 1
    
    return result

# Verify: 3G computed two ways
G3_algo = scalar_mult(3, G)
G3_manual = point_add(G, point_double(G))
print(f"3G (double-and-add): {G3_algo}")
print(f"3G (G + 2G):         {G3_manual}")
print(f"Match: {G3_algo == G3_manual}  ✓")

# The fundamental property: N × G = O (wraps around)
# (Don't actually compute this — it would take forever)
# But we can verify: (N-1)×G + G = O
print(f"\nFundamental: N × G = O (point at infinity)")
print(f"This means private key space is cyclic with order N.")
print(f"N ≈ 1.16 × 10⁷⁷ — more than atoms in the observable universe.")

def serialize_compressed(p: Point) -> bytes:
    """Point → 33-byte compressed public key."""
    prefix = 0x03 if (p.y & 1) else 0x02
    return bytes([prefix]) + p.x.to_bytes(32, 'big')

def parse_compressed(data: bytes) -> Point:
    """33-byte compressed public key → Point."""
    x = int.from_bytes(data[1:], 'big')
    y2 = (pow(x, 3, SECP_P) + 7) % SECP_P
    y = mod_sqrt(y2, SECP_P)
    if (y & 1) != (data[0] == 0x03):
        y = SECP_P - y
    return Point(x, y)

# Demo: generate a key pair
import secrets
private_key = secrets.randbelow(SECP_N - 1) + 1
public_key = scalar_mult(private_key, G)
compressed = serialize_compressed(public_key)

print(f"=== Key Pair Generation ===")
print(f"Private key (d):  {hex(private_key)[:20]}...")
print(f"Public key (P = d×G):")
print(f"  x: {hex(public_key.x)}")
print(f"  y: {hex(public_key.y)}")
print(f"Compressed: {compressed.hex()}")
print(f"  Prefix 0x{compressed[0]:02x} → y is {'odd' if compressed[0] == 0x03 else 'even'}")

# Round-trip verification
recovered = parse_compressed(compressed)
print(f"\nRound-trip: {public_key == recovered}  ✓")


# Module 5: ECDSA — Digital Signatures

This is where everything we've built pays off.

In the intro we said the one-way function fires twice in Bitcoin. Now we have
the math to see exactly what that means:

- **Key generation** (happened once, when the wallet was created): you picked a
  secret integer $d$, computed $P = d \times G$, and published $P$ as your
  address. The one-way function is why nobody can reverse your address back to
  your private key — they'd have to solve the discrete log problem on secp256k1.

- **Signing** (happening right now, for this transaction): you pick a fresh
  random $k$, compute $R = k \times G$, and use $k$ together with $d$ and the
  message hash to produce the signature. The one-way function fires *again* —
  hiding $k$ inside $R$. If anyone could reverse $R$ back to $k$, they could
  solve for $d$ algebraically from the signing equation.

Both uses rely on the same hardness: given a point and the generator, you
cannot find the scalar. Everything below is the machinery that makes this work.

## 5.1 Signature Generation

Given private key $d$, message hash $z$:

1. Pick random nonce $k$
2. Compute $R = k \times G$ and let $r = R_x \bmod N$
3. Compute $s = k^{-1}(z + r \cdot d) \bmod N$
4. Signature is $(r, s)$

## 5.2 Signature Verification

Given public key $P$, message hash $z$, signature $(r, s)$:

1. Compute $u_1 = z \cdot s^{-1} \bmod N$
2. Compute $u_2 = r \cdot s^{-1} \bmod N$
3. Compute $R' = u_1 \times G + u_2 \times P$
4. Verify: $R'_x \bmod N \stackrel{?}{=} r$

In [20]:
import hashlib

def ecdsa_sign(message: bytes, private_key: int) -> tuple:
    """Generate ECDSA signature (r, s)."""
    z = int.from_bytes(hashlib.sha256(message).digest(), 'big')
    
    while True:
        k = secrets.randbelow(SECP_N - 1) + 1
        R = scalar_mult(k, G)
        r = R.x % SECP_N
        if r == 0:
            continue
        
        k_inv = pow(k, SECP_N - 2, SECP_N)
        s = (k_inv * (z + r * private_key)) % SECP_N
        if s == 0:
            continue
        
        return (r, s)

def ecdsa_verify(message: bytes, signature: tuple, public_key: Point) -> bool:
    """Verify ECDSA signature."""
    r, s = signature
    z = int.from_bytes(hashlib.sha256(message).digest(), 'big')
    
    s_inv = pow(s, SECP_N - 2, SECP_N)
    u1 = (z * s_inv) % SECP_N
    u2 = (r * s_inv) % SECP_N
    
    R_prime = point_add(scalar_mult(u1, G), scalar_mult(u2, public_key))
    
    return R_prime.x % SECP_N == r

# Generate key pair
d = secrets.randbelow(SECP_N - 1) + 1
P = scalar_mult(d, G)

# Sign a message
msg = b"Hello, Bitcoin!"
sig = ecdsa_sign(msg, d)
r, s = sig

print("=== ECDSA Demonstration ===")
print(f"Message: {msg.decode()}")
print(f"Private key d: {hex(d)[:20]}...")
print(f"Public key P:  {P}")
print(f"\nSignature:")
print(f"  r = {hex(r)[:20]}...")
print(f"  s = {hex(s)[:20]}...")

# Verify
valid = ecdsa_verify(msg, sig, P)
print(f"\nVerification: {valid}  ✓")

# Tamper with message
tampered = b"Hello, Bitcorn!"
valid_tampered = ecdsa_verify(tampered, sig, P)
print(f"Tampered msg verification: {valid_tampered}  ✗ (correctly rejected)")

# Wrong key
wrong_key = scalar_mult(secrets.randbelow(SECP_N - 1) + 1, G)
valid_wrong = ecdsa_verify(msg, sig, wrong_key)
print(f"Wrong key verification: {valid_wrong}  ✗ (correctly rejected)")

=== ECDSA Demonstration ===
Message: Hello, Bitcoin!
Private key d: 0x62803095ae0f3e18f0...
Public key P:  (0x1a6fd47163f9d1..., 0xeff9a08f84c1d0...)

Signature:
  r = 0xb590fe4dcf4f6a53ce...
  s = 0xeeb1d38e2a4f365cfa...

Verification: True  ✓
Tampered msg verification: False  ✗ (correctly rejected)
Wrong key verification: False  ✗ (correctly rejected)


## 5.3 Why Verification Works — The Proof

The verifier computes $R' = u_1 \times G + u_2 \times P$. Let's prove $R' = R$:

$$R' = u_1 G + u_2 P$$

Substitute $u_1 = z/s$, $u_2 = r/s$, and $P = dG$:

$$R' = \frac{z}{s} G + \frac{r}{s}(dG)$$

$$= \frac{z + rd}{s} G$$

From signing: $s = k^{-1}(z + rd)$, so $k = (z + rd)/s$:

$$R' = k G = R \quad \checkmark$$

## 5.4 The Nonce Catastrophe

If the same nonce $k$ is used for two different messages, the private key leaks:

$$s_1 = k^{-1}(z_1 + r \cdot d) \quad s_2 = k^{-1}(z_2 + r \cdot d)$$

$$s_1 - s_2 = k^{-1}(z_1 - z_2) \implies k = \frac{z_1 - z_2}{s_1 - s_2}$$

Once $k$ is known: $d = r^{-1}(sk - z)$. **Game over.**

This is exactly what happened to Sony's PS3 signing key in 2010.

---

In [21]:
# Demonstration: nonce reuse catastrophe

victim_priv = secrets.randbelow(SECP_N - 1) + 1
victim_pub = scalar_mult(victim_priv, G)

# Victim signs two messages with the SAME nonce (fatal mistake)
k_reused = secrets.randbelow(SECP_N - 1) + 1
R_k = scalar_mult(k_reused, G)
r_val = R_k.x % SECP_N
k_inv = pow(k_reused, SECP_N - 2, SECP_N)

msg1 = b"Transfer 1 BTC to Alice"
msg2 = b"Transfer 2 BTC to Bob"
z1 = int.from_bytes(hashlib.sha256(msg1).digest(), 'big')
z2 = int.from_bytes(hashlib.sha256(msg2).digest(), 'big')

s1 = (k_inv * (z1 + r_val * victim_priv)) % SECP_N
s2 = (k_inv * (z2 + r_val * victim_priv)) % SECP_N

print("=== Nonce Reuse Attack ===")
print(f"Attacker sees two signatures with same r:")
print(f"  sig1: r = {hex(r_val)[:16]}..., s1 = {hex(s1)[:16]}...")
print(f"  sig2: r = {hex(r_val)[:16]}..., s2 = {hex(s2)[:16]}...")
print(f"  Same r → same nonce k was used!")

# Attacker recovers k
k_recovered = ((z1 - z2) * pow(s1 - s2, SECP_N - 2, SECP_N)) % SECP_N
print(f"\nAttacker recovers k: {k_recovered == k_reused}")

# Attacker recovers private key
d_recovered = (pow(r_val, SECP_N - 2, SECP_N) * (s1 * k_recovered - z1)) % SECP_N
print(f"Attacker recovers private key: {d_recovered == victim_priv}")
print(f"\n⚠️  NEVER reuse a nonce. Bitcoin uses RFC 6979 (deterministic k).")

=== Nonce Reuse Attack ===
Attacker sees two signatures with same r:
  sig1: r = 0xba8ac0484dc289..., s1 = 0xe51a62bbdd9965...
  sig2: r = 0xba8ac0484dc289..., s2 = 0x549c8f1893d495...
  Same r → same nonce k was used!

Attacker recovers k: True
Attacker recovers private key: True

⚠️  NEVER reuse a nonce. Bitcoin uses RFC 6979 (deterministic k).


## 5.5 From Math to Bytes

The ECDSA math produces $(r, s)$ as big integers. But Bitcoin transactions are
byte streams sent over a network. How does the math become wire format?

### ECDSA signatures: DER encoding

ECDSA signatures use **DER** (Distinguished Encoding Rules), a variable-length
ASN.1 format:

```
0x30 || total_len || 0x02 || r_len || r_bytes || 0x02 || s_len || s_bytes
```

Each integer ($r$ and $s$) is encoded as a variable-length byte string — if the
high bit is set, a `0x00` padding byte is prepended to keep it positive. This
makes ECDSA signatures **variable length** (typically 71–73 bytes), plus a
sighash flag byte appended at the end.

**Low-S normalization (BIP 62):** Bitcoin requires $s \leq N/2$. If $s > N/2$,
replace it with $N - s$. This eliminates a form of **transaction malleability**
— without it, anyone could flip $s$ to $N - s$ and change the transaction's
hash without invalidating the signature.

### Schnorr signatures: fixed 64 bytes

Schnorr (BIP 340) is simpler: **fixed 64 bytes**, just `R_x (32 bytes) || s (32 bytes)`.
No DER, no variable lengths, no padding, no malleability. This is what Taproot
transactions use.

| Format | Encoding | Size | Malleability |
|--------|----------|------|-------------|
| ECDSA | DER (variable) | ~71–73 bytes + sighash | Requires low-S fix (BIP 62) |
| Schnorr | Fixed `R_x \|\| s` | 64 bytes | None by design |

### Where does the signature live in a transaction?

A Bitcoin transaction is a byte stream with this layout:

```
 SegWit Transaction (BIP 141)
┌──────────┬────────┬──────┬────────┬─────────┬─────────┬──────────┐
│ version  │ marker │ flag │ inputs │ outputs │ witness │ locktime │
│ 4 bytes  │  0x00  │ 0x01 │  var   │   var   │   var   │ 4 bytes  │
└──────────┴────────┴──────┴────────┴─────────┴─────────┴──────────┘
```

The signature's location depends on the script type:

| Script type | Where the signature goes | What else is included |
|-------------|------------------------|----------------------|
| **P2PKH** (legacy) | `scriptSig` field of the input | compressed public key (33 bytes) |
| **P2WPKH** (SegWit v0) | `witness` field (scriptSig is empty) | compressed public key (33 bytes) |
| **P2TR key-path** (Taproot) | `witness` field — just the 64-byte signature | nothing else needed |
| **P2TR script-path** (Taproot) | `witness` field — signatures + script + control block | tapscript, internal key, parity bit |

For Taproot key-path spends, the witness is minimal — a single stack item:

```
 P2TR Key-Path Witness
┌───────────────┬──────────────┬──────────────────┐
│ stack items: 1│ item len: 64 │ signature (64 B)  │
│    0x01       │    0x40      │  R_x || s         │
└───────────────┴──────────────┴──────────────────┘
```

In [ ]:
def der_encode_integer(value: int) -> bytes:
    """Encode a positive integer in DER format (0x02 || length || bytes)."""
    b = value.to_bytes((value.bit_length() + 7) // 8, 'big')
    if b[0] & 0x80:
        b = b'\x00' + b
    return bytes([0x02, len(b)]) + b

def der_encode_signature(r: int, s: int) -> bytes:
    """DER-encode an ECDSA signature: 0x30 || total_len || r_der || s_der."""
    r_der = der_encode_integer(r)
    s_der = der_encode_integer(s)
    body = r_der + s_der
    return bytes([0x30, len(body)]) + body

def low_s_normalize(s: int, N: int) -> int:
    """BIP 62: if s > N/2, replace with N - s."""
    if s > N // 2:
        return N - s
    return s

# Use the signature we generated in section 5.2
# r and s are the two components of any ECDSA signature
r_example = 0xDEADBEEF_CAFEBABE_12345678_9ABCDEF0_DEADBEEF_CAFEBABE_12345678_9ABCDEF0
s_example = 0xFEDCBA98_76543210_FEDCBA98_76543210_FEDCBA98_76543210_FEDCBA98_76543210

print("=== DER Encoding (ECDSA) ===\n")
der_sig = der_encode_signature(r_example, s_example)
print(f"r = {hex(r_example)[:20]}...")
print(f"s = {hex(s_example)[:20]}...")
print(f"\nDER-encoded: {der_sig.hex()}")
print(f"DER length:  {len(der_sig)} bytes (variable)")
print(f"\nByte breakdown:")
print(f"  0x30       = SEQUENCE tag")
print(f"  0x{der_sig[1]:02x}       = total body length ({der_sig[1]} bytes)")
print(f"  0x02       = INTEGER tag (r)")
print(f"  0x{der_sig[3]:02x}       = r length ({der_sig[3]} bytes)")
print(f"  {der_sig[4:4+der_sig[3]].hex()[:40]}... = r value")
r_end = 4 + der_sig[3]
print(f"  0x02       = INTEGER tag (s)")
print(f"  0x{der_sig[r_end+1]:02x}       = s length ({der_sig[r_end+1]} bytes)")
print(f"  {der_sig[r_end+2:].hex()[:40]}... = s value")

# With sighash byte (SIGHASH_ALL = 0x01)
der_with_sighash = der_sig + bytes([0x01])
print(f"\n+ sighash byte (0x01) = {len(der_with_sighash)} bytes total on the wire")

# Low-S normalization
print(f"\n=== Low-S Normalization (BIP 62) ===\n")
s_high = SECP_N - 1
s_low = low_s_normalize(s_high, SECP_N)
print(f"Original s:   {hex(s_high)[:20]}... (> N/2)")
print(f"Normalized s: {hex(s_low)[:20]}... (= N - s)")
print(f"s > N/2: {s_high > SECP_N // 2} → replaced to prevent malleability")

# Schnorr comparison
print(f"\n=== Schnorr Format (BIP 340) ===\n")
R_x = r_example.to_bytes(32, 'big')
s_bytes = s_example.to_bytes(32, 'big')
schnorr_sig = R_x + s_bytes
print(f"R_x (32 bytes) || s (32 bytes) = {len(schnorr_sig)} bytes (fixed)")
print(f"Schnorr: {schnorr_sig.hex()[:40]}...{schnorr_sig.hex()[-8:]}")

# P2TR key-path witness
print(f"\n=== P2TR Key-Path Witness ===\n")
witness_stack_items = bytes([0x01])
witness_sig_len = bytes([0x40])
witness = witness_stack_items + witness_sig_len + schnorr_sig
print(f"Stack items: 1 (0x01)")
print(f"Item length: 64 (0x40)")
print(f"Signature:   {schnorr_sig.hex()[:20]}...")
print(f"Total witness: {len(witness)} bytes")
print(f"\nCompare: ECDSA DER ≈ {len(der_with_sighash)} bytes, Schnorr = {len(schnorr_sig)} bytes")